# REST API Extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/apis/rest_api/rest_api_example.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/apis/rest_api/rest_api_example.ipynb)

## Business Scenario

Teams pull data from external APIs such as weather, payments, or CRM. Without a contract, pagination and schema drift break pipelines.

## Value Proposition

- Contract-driven API extraction with consistent schema
- Reusable auth and pagination patterns
- Validated data before landing in the lakehouse

---

## Goals

1. Configure a REST API contract
2. Run extraction with LakeLogic
3. Inspect validated output


##  Setup

Install API support.

In [1]:
from pathlib import Path
import json
import urllib.request
from lakelogic import DataProcessor

BASE = Path.cwd()
contract_path = BASE / "weather_contract.yaml"
if not contract_path.exists():
    candidate = BASE / "examples" / "03_data_sources" / "apis" / "rest_api" / "weather_contract.yaml"
    if candidate.exists():
        BASE = candidate.parent
        contract_path = candidate

def fetch_weather_rows():
    url = (
        "https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41"
        "&hourly=temperature_2m"
    )
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            payload = json.load(resp)
    except Exception as exc:
        print(f"API request failed, using sample data: {exc}")
        payload = {
            "hourly": {
                "time": ["2026-02-15T00:00", "2026-02-15T01:00"],
                "temperature_2m": [2.1, 1.8],
            }
        }

    times = payload.get("hourly", {}).get("time", [])
    temps = payload.get("hourly", {}).get("temperature_2m", [])
    return [
        {"time": t, "temperature_2m": temp}
        for t, temp in zip(times, temps)
    ]

rows = fetch_weather_rows()
processor = DataProcessor(contract=contract_path)
result = processor.run(rows, source_path="open-meteo")

try:
    processor.materialize(result.good, result.bad)
except Exception as exc:
    print(f"Materialization failed: {exc}")

print(result)

output_file = BASE / "data" / "bronze" / "weather" / "data.parquet"
if output_file.exists():
    viewer = DataProcessor(
        contract={
            "version": "1.0.0",
            "dataset": "weather_view",
            "source": {"type": "table", "path": str(output_file)},
        }
    )
    view_result = viewer.run_source()
    print("Sample output:")
    print(view_result.good)
else:
    print(f"Output not found: {output_file.resolve()}")


2026-02-15 04:00:39.329 | INFO     | lakelogic.core.processor:run:278 - Starting LakeLogic run [Auto-Engine: polars, Contract: berlin_weather]


2026-02-15 04:00:39.338 | INFO     | lakelogic.core.processor:run:319 - Run complete. Source: 168, Total (post-transform): 168, Good: 168, Quarantined: 0, Pre-Transform Dropped: 0, Ratio: 0.00%


2026-02-15 04:00:40.390 | INFO     | lakelogic.core.materialization:materialize_dataframe:984 - Materialized 168 rows to D:\Github\_SaaS\lakelogic\examples\03_data_sources\apis\rest_api\data\bronze\weather\data.parquet


2026-02-15 04:00:40.397 | INFO     | lakelogic.core.processor:run_source:408 - Loading source: D:\Github\_SaaS\lakelogic\examples\03_data_sources\apis\rest_api\data\bronze\weather\data.parquet via polars


2026-02-15 04:00:40.406 | INFO     | lakelogic.core.processor:run:278 - Starting LakeLogic run [Auto-Engine: polars, Contract: weather_view]


2026-02-15 04:00:40.410 | INFO     | lakelogic.core.processor:run:319 - Run complete. Source: 168, Total (post-transform): 168, Good: 168, Quarantined: 0, Pre-Transform Dropped: 0, Ratio: 0.00%


ValidationResult(good=168, bad=0, raw=?)
Sample output:
shape: (168, 2)
┌──────────────────┬────────────────┐
│ time             ┆ temperature_2m │
│ ---              ┆ ---            │
│ str              ┆ f64            │
╞══════════════════╪════════════════╡
│ 2026-02-15T00:00 ┆ -1.8           │
│ 2026-02-15T01:00 ┆ -1.8           │
│ 2026-02-15T02:00 ┆ -1.9           │
│ 2026-02-15T03:00 ┆ -2.0           │
│ 2026-02-15T04:00 ┆ -2.2           │
│ …                ┆ …              │
│ 2026-02-21T19:00 ┆ 4.6            │
│ 2026-02-21T20:00 ┆ 4.9            │
│ 2026-02-21T21:00 ┆ 5.2            │
│ 2026-02-21T22:00 ┆ 5.5            │
│ 2026-02-21T23:00 ┆ 5.9            │
└──────────────────┴────────────────┘


##  Run the Extraction

Pass the YAML contract path directly to the `DataProcessor`.

##  Verify Results

LakeLogic stores data in Parquet format by default.